# 06 · User seen vs unseen: SMLP4Rec plain vs hybrid (SMLP4Rec + prior + sameDest)

Notebook chỉ test: **không huấn luyện, không tinh chỉnh**. Nó nạp lại checkpoint tốt nhất của notebook 01 và trọng số hybrid đã chọn của notebook 04 (`results/week3_implementation/smlprec_expedia_hybrid_samedest.json`), chấm lại valid và test, và chia các sự kiện theo việc user có trong tập huấn luyện hay không.

**User seen vs unseen**

- **Seen**: user có ít nhất một dòng trong **tập chia train** của RecBole (một mục tiêu train: booking trong cửa sổ train mà không phải booking đầu tiên của user). Mô hình đã được huấn luyện trên các mục tiêu next-booking của user này.
- **Unseen (cold-start)**: không có dòng train. Hai loại sự kiện:
  - **unseen, L ≥ 1**: user có booking trước đó, nhưng tất cả (trừ có thể một booking đầu tiên) đều nằm sau cửa sổ train, nên lịch sử mà mô hình đọc lúc test chưa bao giờ là nhãn huấn luyện.
  - **unseen, L = 0**: booking đầu tiên của user (không có lịch sử). Đây là các "dòng cold" của notebook 03/04.
- Mọi dòng L = 0 đều unseen theo cấu tạo (được assert bên dưới).

**Các phương pháp (giống notebook 03/04, trọng số cố định)**

| | dòng warm (L ≥ 1) | dòng cold (L = 0) |
|---|---|---|
| SMLP4Rec plain | softmax của mô hình trên 100 cluster | global popularity (mô hình cần ≥ 1 booking trước đó) |
| hybrid | `log q_SMLP4Rec + w_p log q_prior + w_s log q_sameDest`, alpha và trọng số từng ô từ notebook 04 (các ô = destination known vs new) | chỉ destination prior |

Metric: Recall@K và NDCG@K (K = 5, 10, 20), xếp hạng đầy đủ trên 100 cluster; khoảng tin cậy 95 % bootstrap theo cặp cho hybrid trừ plain (Recall@5, NDCG@5).

Kernel: `smlp4rec`. Cần `data/interim/recbole/expedia_dest/expedia_dest.inter` và `train.csv` thô (các bảng prior).

In [1]:
import functools
import json
import logging
import sys
import time
from pathlib import Path

import numpy as np
import torch  # before pandas (Windows c10.dll issue)

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "src")]
torch.load = functools.partial(torch.load, weights_only=False)

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import init_seed

from nbp.eval.bootstrap import paired_bootstrap
from nbp.eval.metrics import summarize, target_rank
from nbp.hybrid.fusion import same_dest_distribution, smoothed_log
from nbp.hybrid.same_dest import same_dest_scores
from src.models.smlprec import SMLPREC

logging.disable(logging.INFO)

CONFIG = ROOT / "configs" / "smlprec_expedia.yaml"
WORK = ROOT / "data" / "interim" / "recbole"
EVENT_COLS = ROOT / "data" / "interim" / "expedia_event_cols_clean.npz"  # row position = event index = TIME
INTER_DEST = WORK / "expedia_clean_dest" / "expedia_clean_dest.inter"
REF_DIR = ROOT / "results" / "week4_rebuild" / "clean_chain"  # outputs and predecessor results on the collapsed data
OLD_DIR = ROOT / "results" / "week3_implementation"  # same chain on the uncollapsed data (reference only)
REF_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = max((ROOT / "experiments").glob("*_expedia_smlp4rec_clean-plain"))

N_CLUSTERS = 100
M_SMOOTH = 5
EPS = 1e-6  # prior floor for cold rows (notebooks 03/04)
TIME_MARGIN = 0  # TIME is the exact event index: no float32 margin
RECENCY = 0.7

HYB_REF = json.loads((REF_DIR / "smlprec_expedia_hybrid_samedest.json").read_text(encoding="utf-8"))
SCHEME = HYB_REF["cohorts"]["chosen_fewest_cells_within_one_se"]
print("scheme chosen by notebook 04 on the collapsed data:", SCHEME)
FINAL_W = {int(k): tuple(v) for k, v in HYB_REF["cohorts"]["final_weights_per_cell"].items()}
ALPHA = HYB_REF["selection"]["one_se_pick"][0]
assert all(w[0] == ALPHA for w in FINAL_W.values())
HYB_NAME = "SMLP4Rec + prior + sameDest (this notebook)"
print("run dir:", RUN_DIR)
print("hybrid weights from notebook 04: alpha", ALPHA, "| cell 0 = new destination, 1 = known destination:", FINAL_W)

scheme chosen by notebook 04 on the collapsed data: known vs new destination (2)
run dir: C:\workspace\PROJECTS\next-best-product-recommendation\experiments\2026-10-08_expedia_smlp4rec_clean-plain
hybrid weights from notebook 04: alpha 0.05 | cell 0 = new destination, 1 = known destination: {0: (0.05, 2.0, 0.0), 1: (0.05, 1.25, 1.0)}


## 1. Dữ liệu, prior và các thành phần điểm (cùng code với notebook 04)

In [2]:
config = Config(
    model=SMLPREC,
    dataset="expedia_clean_dest",
    config_file_list=[str(CONFIG)],
    config_dict={
        "data_path": str(WORK),
        "checkpoint_dir": str(RUN_DIR),
        "load_col": {"inter": ["user_id", "item_id", "timestamp", "srch_destination_id"]},
    },
)
init_seed(config["seed"], config["reproducibility"])
t0 = time.time()
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)
tok2id = dataset.field2token_id[dataset.iid_field]
uid_of_token = dataset.field2token_id[dataset.uid_field]
cluster_ids = np.array([tok2id[str(k)] for k in range(N_CLUSTERS)])  # model column of each raw cluster
col_of_model_id = {int(m): k for k, m in enumerate(cluster_ids)}
ITEM_SEQ = config["ITEM_ID_FIELD"] + config["LIST_SUFFIX"]
ITEM_SEQ_LEN = config["ITEM_LIST_LENGTH_FIELD"]
TIME = config["TIME_FIELD"]
UID = config["USER_ID_FIELD"]


def split_arrays(loader) -> dict:
    d = loader.dataset.inter_feat
    seq, length = d[ITEM_SEQ].numpy(), d[ITEM_SEQ_LEN].numpy()
    y_model = d[config["ITEM_ID_FIELD"]].numpy()
    q_dest = d["srch_destination_id"].numpy().astype(np.int64)
    h_dest = d["srch_destination_id" + config["LIST_SUFFIX"]].numpy().astype(np.int64)
    pos = np.arange(seq.shape[1])[None, :] < length[:, None]
    return {
        "seq": seq,
        "seq_t": d[ITEM_SEQ],
        "length": length,
        "length_t": d[ITEM_SEQ_LEN],
        "y": np.array([col_of_model_id[int(t)] for t in y_model]),  # raw cluster index 0..99
        "user": d[UID].numpy(),
        "q_dest": q_dest,
        "h_dest": h_dest,
        "q_seen": ((h_dest == q_dest[:, None]) & pos).any(1),
    }


arr = {"valid": split_arrays(valid_data), "test": split_arrays(test_data)}
train_users = np.unique(train_data.dataset.inter_feat[UID].numpy())
print({sp: len(a["y"]) for sp, a in arr.items()}, "| users with a train row:", len(train_users), f"| {time.time() - t0:.0f}s")

{'valid': 213869, 'test': 213869} | users with a train row: 415354 | 53s


In [3]:
import pandas as pd  # after torch

t_train_max = float(train_data.dataset.inter_feat[TIME].max())
t_valid_max = float(valid_data.dataset.inter_feat[TIME].max())
cut = t_train_max - TIME_MARGIN


def build_prior_tables(cut_event: float) -> dict:
    """Cluster count tables from the bookings strictly before event index cut_event (collapsed data)."""
    z, k = np.load(EVENT_COLS), int(cut_event)
    b = pd.DataFrame({"srch_destination_id": z["dest"][:k], "hotel_market": z["market"][:k], "hotel_cluster": z["item"][:k]}).astype("int64")

    def table(key):
        t = b.groupby([key, "hotel_cluster"]).size().unstack(fill_value=0)
        return t.reindex(columns=range(N_CLUSTERS), fill_value=0)

    dest, mkt = table("srch_destination_id"), table("hotel_market")
    dm = b.groupby(["srch_destination_id", "hotel_market"]).size().reset_index(name="n")
    modal = dm.sort_values("n", ascending=False).drop_duplicates("srch_destination_id")
    glob = np.bincount(b["hotel_cluster"].values, minlength=N_CLUSTERS).astype(np.float64)
    return {
        "n_bookings": len(b),
        "dest": {int(d): r.astype(np.float64) for d, r in zip(dest.index, dest.values)},
        "market": {int(m): r.astype(np.float64) for m, r in zip(mkt.index, mkt.values)},
        "dest2mkt": dict(zip(modal["srch_destination_id"].astype(int), modal["hotel_market"].astype(int))),
        "global": glob / glob.sum(),
    }


def prior_vector(d: int) -> np.ndarray:
    """p(cluster | destination), shape (100,), raw cluster order."""
    row = tables["dest"].get(d)
    if row is None:
        return tables["global"]
    pm = tables["market"][tables["dest2mkt"][d]]
    pm = pm / pm.sum()
    return (row + M_SMOOTH * pm) / (row.sum() + M_SMOOTH)


def prior_prob(dests: np.ndarray) -> np.ndarray:
    """(n, 100) prior distribution of each query destination, raw cluster order."""
    uniq, inv = np.unique(dests, return_inverse=True)
    return np.stack([prior_vector(int(d)) for d in uniq]).astype(np.float32)[inv]


t1 = time.time()
tables = build_prior_tables(cut)
print(f"prior cut (unix s) {cut:.0f}; {tables['n_bookings']} bookings, {len(tables['dest'])} destinations; {time.time() - t1:.0f}s")

model = SMLPREC(config, dataset)
model.load_state_dict(torch.load(RUN_DIR / "best.pth")["state_dict"])


@torch.no_grad()
def model_prob(a: dict, batch: int = 8192) -> np.ndarray:
    """(n, 100) softmax of the model over the 100 clusters (padding column excluded), raw cluster order."""
    model.eval()
    out = []
    for i in range(0, len(a["y"]), batch):
        s = model.full_sort_predict(
            {model.ITEM_SEQ: a["seq_t"][i : i + batch], model.ITEM_SEQ_LEN: a["length_t"][i : i + batch]}
        )
        s[:, 0] = float("-inf")
        out.append(torch.softmax(s, dim=1).numpy()[:, cluster_ids])
    return np.concatenate(out)


t2 = time.time()
p_model, p_prior, sd = {}, {}, {}
for sp, a in arr.items():
    p_model[sp], p_prior[sp] = model_prob(a), prior_prob(a["q_dest"])
    sd[sp] = same_dest_scores(
        a["seq"], a["length"], a["h_dest"], a["q_dest"], dataset.item_num, base=RECENCY, normalize=True
    )[:, cluster_ids]
print(f"components ready in {time.time() - t2:.0f}s", {sp: p_model[sp].shape for sp in arr})

prior cut (unix s) 2424705; 2424705 bookings, 34507 destinations; 1s


components ready in 13s {'valid': (213869, 100), 'test': (213869, 100)}


## 2. Dòng cold (L = 0)

Booking đầu tiên của mỗi user, trong cửa sổ valid `[mục tiêu train cuối, mục tiêu valid cuối)` và cửa sổ test `[mục tiêu valid cuối, hết]`, như trong notebook 03/04.

In [4]:
inter = pd.read_csv(INTER_DEST, sep="\t").sort_values(["user_id:token", "timestamp:float"], kind="stable")
first = inter.drop_duplicates("user_id:token", keep="first")
windows = {
    "valid": (first["timestamp:float"] >= t_train_max) & (first["timestamp:float"] < t_valid_max),
    "test": first["timestamp:float"] >= t_valid_max,
}
cold = {}
for sp, mask in windows.items():
    f = first[mask]
    cold[sp] = {
        "y": f["item_id:token"].astype(int).values,  # raw cluster index 0..99
        "dest": f["srch_destination_id:float"].values.astype(np.int64),
        "user": np.array([uid_of_token[str(u)] for u in f["user_id:token"]]),
    }
    assert not np.isin(cold[sp]["user"], train_users).any(), "a first booking user has a train row"
print({sp: len(c["y"]) for sp, c in cold.items()})

{'valid': 50769, 'test': 49462}


## 3. Seen vs unseen: bao nhiêu user và sự kiện

Số đếm theo từng tập chia, trên **mọi sự kiện** của tập đó (dòng warm + dòng cold). Một user được đếm một lần cho mỗi tập chia; "% user" tính trên số user khác nhau có ít nhất một sự kiện trong tập đó.

In [5]:
seen_rows = {sp: np.isin(arr[sp]["user"], train_users) for sp in arr}


def presence_stats(sp: str) -> dict:
    w_user, c_user, w_seen = arr[sp]["user"], cold[sp]["user"], seen_rows[sp]
    users = np.unique(np.concatenate([w_user, c_user]))
    seen_u = np.isin(users, train_users)
    n_rows = len(w_user) + len(c_user)
    unseen_warm_users = np.unique(w_user[~w_seen])
    return {
        "users": int(len(users)),
        "seen_users": int(seen_u.sum()),
        "unseen_users": int((~seen_u).sum()),
        "seen_users_pct": round(100 * float(seen_u.mean()), 2),
        "unseen_users_pct": round(100 * float((~seen_u).mean()), 2),
        "unseen_users_with_history_(L>=1)": int(len(unseen_warm_users)),
        "unseen_users_first_booking_only_(L=0)": int(len(np.setdiff1d(c_user, unseen_warm_users))),
        "events": int(n_rows),
        "seen_events": int(w_seen.sum()),
        "unseen_events": int((~w_seen).sum() + len(c_user)),
        "seen_events_pct": round(100 * float(w_seen.sum()) / n_rows, 2),
        "unseen_events_pct": round(100 * float((~w_seen).sum() + len(c_user)) / n_rows, 2),
        "unseen_events_L>=1": int((~w_seen).sum()),
        "unseen_events_L=0": int(len(c_user)),
        "mean_history_len_seen": round(float(arr[sp]["length"][w_seen].mean()), 3),
        "mean_history_len_unseen_L>=1": round(float(arr[sp]["length"][~w_seen].mean()), 3),
    }


presence = {sp: presence_stats(sp) for sp in ("valid", "test")}
for sp in presence:
    p = presence[sp]
    assert p["seen_users"] + p["unseen_users"] == p["users"]
    assert p["seen_events"] + p["unseen_events"] == p["events"]
pd.DataFrame(presence)

,valid,test
users,166388.000,165604.000
seen_users,88168.000,80645.000
unseen_users,78220.000,84959.000
seen_users_pct,52.990,48.700
unseen_users_pct,47.010,51.300
unseen_users_with_history_(L>=1),38671.000,47710.000
unseen_users_first_booking_only_(L=0),39549.000,37249.000
events,264638.000,263331.000
seen_events,157601.000,139815.000
unseen_events,107037.000,123516.000


## 4. Chấm điểm cả hai phương pháp (trọng số cố định) và đối chiếu với các lượt chạy đã lưu

- SMLP4Rec plain trên dòng warm phải tái tạo notebook 01 (`metrics.json` của lượt chạy).
- SMLP4Rec plain warm + global popularity cho cold phải tái tạo "SMLP4Rec plain trên warm + global popularity trên cold" của notebook 03.
- Hybrid phải tái tạo số test và valid của notebook 04 (warm và all events).

In [6]:
def rank(scores: np.ndarray, y: np.ndarray) -> np.ndarray:
    return target_rank(scores, y)


PLAIN = "plain SMLP4Rec"
HYB = "hybrid (SMLP4Rec + prior + sameDest)"
ranks = {sp: {} for sp in arr}
for sp, a in arr.items():
    cell = a["q_seen"].astype(int)  # scheme "known vs new destination": 1 = known
    wp = np.array([FINAL_W[c][1] for c in cell], np.float32)[:, None]
    ws = np.array([FINAL_W[c][2] for c in cell], np.float32)[:, None]
    M = smoothed_log(p_model[sp], ALPHA)
    P = smoothed_log(p_prior[sp], ALPHA)
    S = smoothed_log(same_dest_distribution(sd[sp]), ALPHA)
    c = cold[sp]
    glob_log = np.tile(np.log(tables["global"] + EPS).astype(np.float32), (len(c["y"]), 1))
    ranks[sp][PLAIN] = {"warm": rank(np.log(p_model[sp]), a["y"]), "cold": rank(glob_log, c["y"])}
    ranks[sp][HYB] = {
        "warm": rank(M + wp * P + ws * S, a["y"]),
        "cold": rank(np.log(prior_prob(c["dest"]) + EPS), c["y"]),
    }
    del M, P, S


def all_events(r: dict) -> np.ndarray:
    return np.concatenate([r["warm"], r["cold"]])


ref01 = json.loads((RUN_DIR / "metrics.json").read_text(encoding="utf-8"))
LF = json.loads((REF_DIR / "smlprec_expedia_late_fusion.json").read_text(encoding="utf-8"))
checks = []
for sp in arr:
    got, want = summarize(ranks[sp][PLAIN]["warm"]), ref01[sp]
    checks.append((f"{sp} plain warm vs notebook 01", got, want))
    got = summarize(all_events(ranks[sp][PLAIN]))
    want = LF["combined_warm_plus_cold"][sp]["plain SMLP4Rec on warm + global popularity on cold"]
    checks.append((f"{sp} plain all events vs notebook 03", got, want))
    for scope in ("warm", "all_events"):
        r = ranks[sp][HYB]["warm"] if scope == "warm" else all_events(ranks[sp][HYB])
        checks.append((f"{sp} hybrid {scope} vs notebook 04", summarize(r), HYB_REF["benchmark"][sp][scope][HYB_NAME]))
MET = ["recall@5", "ndcg@5", "recall@10", "ndcg@10", "recall@20", "ndcg@20"]
for name, got, want in checks:
    same = all(got[m] == want[m] for m in MET if m in want)
    print(f"{name:42s} {'OK' if same else 'MISMATCH'}  R@5 {got['recall@5']} vs {want['recall@5']}  N@5 {got['ndcg@5']} vs {want.get('ndcg@5')}")
    assert same, name

valid plain warm vs notebook 01            OK  R@5 0.3204 vs 0.3204  N@5 0.2277 vs 0.2277
valid plain all events vs notebook 03      OK  R@5 0.2829 vs 0.2829  N@5 0.1994 vs 0.1994
valid hybrid warm vs notebook 04           OK  R@5 0.6111 vs 0.6111  N@5 0.448 vs 0.448
valid hybrid all_events vs notebook 04     OK  R@5 0.5955 vs 0.5955  N@5 0.4314 vs 0.4314
test plain warm vs notebook 01             OK  R@5 0.3212 vs 0.3212  N@5 0.2274 vs 0.2274
test plain all events vs notebook 03       OK  R@5 0.2856 vs 0.2856  N@5 0.2 vs 0.2
test hybrid warm vs notebook 04            OK  R@5 0.6028 vs 0.6028  N@5 0.4419 vs 0.4419
test hybrid all_events vs notebook 04      OK  R@5 0.5858 vs 0.5858  N@5 0.4248 vs 0.4248


## 5. Kết quả: SMLP4Rec plain vs hybrid trên user seen và unseen

Các tập con của sự kiện từng tập chia (warm + cold):

- **user seen**: dòng warm của user có dòng train (không dòng cold nào có thể là seen).
- **user unseen**: dòng warm của user không có dòng train + mọi dòng cold. Chia nhỏ thành **unseen, L ≥ 1** và **unseen, L = 0 (booking đầu tiên)**, vì mô hình plain quay về global popularity ở L = 0.
- **all events**: tham chiếu (bằng notebook 03/04).

In [7]:
SUBSETS = ["seen users", "unseen users", "unseen, L>=1", "unseen, L=0 (first booking)", "all events"]


def subset_ranks(sp: str, name: str) -> dict:
    r, s = ranks[sp][name], seen_rows[sp]
    return {
        "seen users": r["warm"][s],
        "unseen users": np.concatenate([r["warm"][~s], r["cold"]]),
        "unseen, L>=1": r["warm"][~s],
        "unseen, L=0 (first booking)": r["cold"],
        "all events": all_events(r),
    }


sub = {sp: {name: subset_ranks(sp, name) for name in (PLAIN, HYB)} for sp in arr}
results = {
    sp: {subset: {name: summarize(sub[sp][name][subset]) for name in (PLAIN, HYB)} for subset in SUBSETS}
    for sp in arr
}


def results_table(sp: str) -> pd.DataFrame:
    rows = {}
    for subset in SUBSETS:
        for name in (PLAIN, HYB):
            rows[(subset, name)] = results[sp][subset][name]
    return pd.DataFrame(rows).T[["n"] + MET].astype({"n": int})


print("TEST")
display(results_table("test"))
print("VALID")
display(results_table("valid"))

TEST


n  \
seen users                  plain SMLP4Rec                        139815   
                            hybrid (SMLP4Rec + prior + sameDest)  139815   
unseen users                plain SMLP4Rec                        123516   
                            hybrid (SMLP4Rec + prior + sameDest)  123516   
unseen, L>=1                plain SMLP4Rec                         74054   
                            hybrid (SMLP4Rec + prior + sameDest)   74054   
unseen, L=0 (first booking) plain SMLP4Rec                         49462   
                            hybrid (SMLP4Rec + prior + sameDest)   49462   
all events                  plain SMLP4Rec                        263331   
                            hybrid (SMLP4Rec + prior + sameDest)  263331   

                                                                  recall@5  \
seen users                  plain SMLP4Rec                          0.3272   
                            hybrid (SMLP4Rec + prior + sameDest)    0.6165   
unseen users                plain SMLP4Rec                          0.2385   
                            hybrid (SMLP4Rec + prior + sameDest)    0.5511   
unseen, L>=1                plain SMLP4Rec                          0.3097   
                            hybrid (SMLP4Rec + prior + sameDest)    0.5768   
unseen, L=0 (first booking) plain SMLP4Rec                          0.1318   
                            hybrid (SMLP4Rec + prior + sameDest)    0.5127   
all events                  plain SMLP4Rec                          0.2856   
                            hybrid (SMLP4Rec + prior + sameDest)    0.5858   

                                                                  ndcg@5  \
seen users                  plain SMLP4Rec                        0.2297   
                            hybrid (SMLP4Rec + prior + sameDest)  0.4540   
unseen users                plain SMLP4Rec                        0.1664   
                            hybrid (SMLP4Rec + prior + sameDest)  0.3918   
unseen, L>=1                plain SMLP4Rec                        0.2231   
                            hybrid (SMLP4Rec + prior + sameDest)  0.4190   
unseen, L=0 (first booking) plain SMLP4Rec                        0.0814   
                            hybrid (SMLP4Rec + prior + sameDest)  0.3510   
all events                  plain SMLP4Rec                        0.2000   
                            hybrid (SMLP4Rec + prior + sameDest)  0.4248   

                                                                  recall@10  \
seen users                  plain SMLP4Rec                           0.4615   
                            hybrid (SMLP4Rec + prior + sameDest)     0.7803   
unseen users                plain SMLP4Rec                           0.3528   
                            hybrid (SMLP4Rec + prior + sameDest)     0.7312   
unseen, L>=1                plain SMLP4Rec                           0.4368   
                            hybrid (SMLP4Rec + prior + sameDest)     0.7509   
unseen, L=0 (first booking) plain SMLP4Rec                           0.2270   
                            hybrid (SMLP4Rec + prior + sameDest)     0.7016   
all events                  plain SMLP4Rec                           0.4105   
                            hybrid (SMLP4Rec + prior + sameDest)     0.7573   

                                                                  ndcg@10  \
seen users                  plain SMLP4Rec                         0.2729   
                            hybrid (SMLP4Rec + prior + sameDest)   0.5071   
unseen users                plain SMLP4Rec                         0.2031   
                            hybrid (SMLP4Rec + prior + sameDest)   0.4501   
unseen, L>=1                plain SMLP4Rec                         0.2640   
                            hybrid (SMLP4Rec + prior + sameDest)   0.4754   
unseen, L=0 (first booking) plain SMLP4Rec                         0.1119   
                            hybrid (SMLP4Rec + prior

VALID


n  \
seen users                  plain SMLP4Rec                        157601   
                            hybrid (SMLP4Rec + prior + sameDest)  157601   
unseen users                plain SMLP4Rec                        107037   
                            hybrid (SMLP4Rec + prior + sameDest)  107037   
unseen, L>=1                plain SMLP4Rec                         56268   
                            hybrid (SMLP4Rec + prior + sameDest)   56268   
unseen, L=0 (first booking) plain SMLP4Rec                         50769   
                            hybrid (SMLP4Rec + prior + sameDest)   50769   
all events                  plain SMLP4Rec                        264638   
                            hybrid (SMLP4Rec + prior + sameDest)  264638   

                                                                  recall@5  \
seen users                  plain SMLP4Rec                          0.3311   
                            hybrid (SMLP4Rec + prior + sameDest)    0.6223   
unseen users                plain SMLP4Rec                          0.2121   
                            hybrid (SMLP4Rec + prior + sameDest)    0.5559   
unseen, L>=1                plain SMLP4Rec                          0.2907   
                            hybrid (SMLP4Rec + prior + sameDest)    0.5799   
unseen, L=0 (first booking) plain SMLP4Rec                          0.1250   
                            hybrid (SMLP4Rec + prior + sameDest)    0.5294   
all events                  plain SMLP4Rec                          0.2829   
                            hybrid (SMLP4Rec + prior + sameDest)    0.5955   

                                                                  ndcg@5  \
seen users                  plain SMLP4Rec                        0.2332   
                            hybrid (SMLP4Rec + prior + sameDest)  0.4579   
unseen users                plain SMLP4Rec                        0.1495   
                            hybrid (SMLP4Rec + prior + sameDest)  0.3925   
unseen, L>=1                plain SMLP4Rec                        0.2123   
                            hybrid (SMLP4Rec + prior + sameDest)  0.4202   
unseen, L=0 (first booking) plain SMLP4Rec                        0.0799   
                            hybrid (SMLP4Rec + prior + sameDest)  0.3618   
all events                  plain SMLP4Rec                        0.1994   
                            hybrid (SMLP4Rec + prior + sameDest)  0.4314   

                                                                  recall@10  \
seen users                  plain SMLP4Rec                           0.4656   
                            hybrid (SMLP4Rec + prior + sameDest)     0.7892   
unseen users                plain SMLP4Rec                           0.3158   
                            hybrid (SMLP4Rec + prior + sameDest)     0.7376   
unseen, L>=1                plain SMLP4Rec                           0.4113   
                            hybrid (SMLP4Rec + prior + sameDest)     0.7569   
unseen, L=0 (first booking) plain SMLP4Rec                           0.2100   
                            hybrid (SMLP4Rec + prior + sameDest)     0.7162   
all events                  plain SMLP4Rec                           0.4050   
                            hybrid (SMLP4Rec + prior + sameDest)     0.7683   

                                                                  ndcg@10  \
seen users                  plain SMLP4Rec                         0.2766   
                            hybrid (SMLP4Rec + prior + sameDest)   0.5120   
unseen users                plain SMLP4Rec                         0.1828   
                            hybrid (SMLP4Rec + prior + sameDest)   0.4513   
unseen, L>=1                plain SMLP4Rec                         0.2510   
                            hybrid (SMLP4Rec + prior + sameDest)   0.4775   
unseen, L=0 (first booking) plain SMLP4Rec                         0.1072   
                            hybrid (SMLP4Rec + prior

In [8]:
boot = {
    subset: paired_bootstrap(sub["test"][HYB][subset], sub["test"][PLAIN][subset], k=5)
    for subset in SUBSETS
}
print("TEST, hybrid minus plain SMLP4Rec, paired bootstrap (1000 resamples, seed 0, 95% CI)")
pd.DataFrame(
    {
        s: {m: f"{v[m]['diff']:+.4f} [{v[m]['ci95'][0]:+.4f}, {v[m]['ci95'][1]:+.4f}]" for m in ("recall@5", "ndcg@5")}
        for s, v in boot.items()
    }
).T

TEST, hybrid minus plain SMLP4Rec, paired bootstrap (1000 resamples, seed 0, 95% CI)


,recall@5,ndcg@5
seen users,"+0.2893 [+0.2865, +0.2920]","+0.2243 [+0.2222, +0.2263]"
unseen users,"+0.3126 [+0.3093, +0.3157]","+0.2254 [+0.2230, +0.2276]"
"unseen, L>=1","+0.2670 [+0.2633, +0.2712]","+0.1959 [+0.1932, +0.1988]"
"unseen, L=0 (first booking)","+0.3809 [+0.3758, +0.3859]","+0.2697 [+0.2658, +0.2733]"
all events,"+0.3002 [+0.2982, +0.3022]","+0.2248 [+0.2234, +0.2263]"


## 6. Lưu

`results/week3_implementation/smlprec_expedia_seen_unseen_users.json` (và một bản sao trong thư mục chạy).

In [9]:
result = {
    "run": "seen vs unseen users: plain SMLP4Rec vs hybrid (SMLP4Rec + prior + sameDest), fixed checkpoint and weights, no retraining",
    "dataset": "expedia (train.csv, is_booking == 1; item = hotel_cluster)",
    "checkpoint": str(RUN_DIR / "best.pth"),
    "definitions": {
        "seen_user": "user has >= 1 row in the RecBole train split (a train target)",
        "unseen_user": "no train row; events = warm rows (L >= 1) of such users + first bookings (L = 0) in the window",
        "plain_SMLP4Rec": "model softmax on L >= 1; global popularity on L = 0",
        "hybrid": "notebook 04 formula and weights on L >= 1; destination prior only on L = 0",
    },
    "hybrid_weights": {"alpha": ALPHA, "scheme": SCHEME, "final_weights_per_cell": {str(k): list(v) for k, v in FINAL_W.items()}},
    "presence": presence,
    "results": results,
    "bootstrap_test_95ci_hybrid_minus_plain": boot,
}
for path in (RUN_DIR / "seen_unseen_users.json", REF_DIR / "smlprec_expedia_seen_unseen_users.json"):
    path.write_text(json.dumps(result, indent=2), encoding="utf-8")
print("saved", REF_DIR / "smlprec_expedia_seen_unseen_users.json")

saved C:\workspace\PROJECTS\next-best-product-recommendation\results\week4_rebuild\clean_chain\smlprec_expedia_seen_unseen_users.json
